# Loading and Preprocessing

In [36]:
from data_loader import load_data

In [37]:
X_clinical,\
X_clinical_A,\
X_clinical_B,\
X_clinical_Delta,\
targets = load_data()

In [38]:
targets.head()

,surv_bestresponse_car,ae_summ_icans_v2,ae_summ_icans_highestgrade_v2,ae_summ_crs_v2,ae_summ_highestgrade_v2
FTC-UMCG-0001,1.0,1.0,1.0,1.0,1.0
FTC-UMCG-0003,4.0,0.0,0.0,1.0,1.0
FTC-UMCG-0004,4.0,1.0,3.0,1.0,2.0
FTC-UMCG-0005,2.0,1.0,2.0,1.0,1.0
FTC-UMCG-0006,1.0,0.0,0.0,1.0,2.0


In [39]:
datasets = {
    "Clinical": X_clinical,
    "Clinical_A": X_clinical_A,
    "Clinical_B": X_clinical_B,
    "Clinical_Delta": X_clinical_Delta
}

# Feature Selection
Dropping highly correlated clinical and radiomics features using the EDA analysis, as they add little to no new information.

## Highly Correlated Clinical Features  
We're using the EDA because it took the mixed feature types into account when calculating the correlations, and there are only a few features, so we could manually write them. So, it was more straight-forward than redoing it ourselves.

In [40]:
clinical_ftrs_to_drop = ['total_num_priortherapylines_fl',
                'tr_car_bridg_type.factor_None',
                'scr_weight',
                'scr_height',
                'indication_age_60',
                'cli_st_neutrophils',
                'indication_extran_invol',
                'indication_pri_refr',
                'indication_sec_refr'
                ]

In [41]:
for name, df in datasets.items():
    print(f"Before dropping features in {name}: {df.shape}")
    df.drop(columns=clinical_ftrs_to_drop, inplace=True)
    print(f"After dropping features in {name}: {df.shape}")

Before dropping features in Clinical: (85, 35)
After dropping features in Clinical: (85, 26)
Before dropping features in Clinical_A: (85, 134)
After dropping features in Clinical_A: (85, 125)
Before dropping features in Clinical_B: (85, 134)
After dropping features in Clinical_B: (85, 125)
Before dropping features in Clinical_Delta: (85, 134)
After dropping features in Clinical_Delta: (85, 125)


## Highly Correlated Radiomic Features
These are all numerical values, with high dimensionality. The EDA already showed a large number of highly correlated features, so we will write code to automatically keep only one of possibly multiple highly correlated features.

In [42]:
import numpy as np

def keep_uncorrelated_features(df, threshold=0.9):
    # Absolute correlation matrix
    corr = df.corr().abs()
    # Upper triangle (ignore diagonal and lower half), k=1 means we start from the first diagonal above the main diagonal
    upper = corr.where(
        np.triu(np.ones(corr.shape), k=1).astype(bool)
    )
    # Find columns with any correlation above threshold
    to_drop = [col for col in upper.columns if (upper[col] > threshold).any() and upper[col].dtype in [np.float64]]
    # Return reduced dataframe
    return df.drop(columns=to_drop)

# Usage
for name, df in datasets.items():
    print(f"Before dropping correlated features in {name}: {df.shape}")

    datasets[name] = keep_uncorrelated_features(df, threshold=0.9)
    print(f"After dropping correlated features in {name}: {datasets[name].shape}")

Before dropping correlated features in Clinical: (85, 26)
After dropping correlated features in Clinical: (85, 26)
Before dropping correlated features in Clinical_A: (85, 125)
After dropping correlated features in Clinical_A: (85, 62)
Before dropping correlated features in Clinical_B: (85, 125)
After dropping correlated features in Clinical_B: (85, 59)
Before dropping correlated features in Clinical_Delta: (85, 125)
After dropping correlated features in Clinical_Delta: (85, 71)


In [43]:
cat_cols = datasets['Clinical'].select_dtypes(exclude='number').columns.tolist()

In [44]:
# Using ColumnTransformer to apply different transformations to numeric and categorical columns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

# Creating a ColumnTransformer to apply StandardScaler to numeric columns and leave categorical columns unchanged
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', 'passthrough', cat_cols)],
        remainder= StandardScaler()
    
)

In [45]:
from model_eval import result_viewer

# ICANS 0,1

In [46]:
y = targets['ae_summ_icans_v2']

In [47]:
y.value_counts()

ae_summ_icans_v2
1.0    48
0.0    37
Name: count, dtype: int64

In [48]:
outcome = result_viewer(preprocessor, datasets, y)

Results for Logistic Regression:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6187 ± 0.1166,0.5680 ± 0.0715,0.5675 ± 0.0618,0.5439 ± 0.1205
train_roc_auc,0.7698 ± 0.0205,0.7582 ± 0.0170,0.7730 ± 0.0373,0.7936 ± 0.0190
test_precision,0.6547 ± 0.0471,0.6141 ± 0.0303,0.5747 ± 0.0387,0.6130 ± 0.1008
train_precision,0.6944 ± 0.0408,0.6847 ± 0.0319,0.7110 ± 0.0367,0.7364 ± 0.0315
test_recall,0.7044 ± 0.1280,0.7933 ± 0.0874,0.8400 ± 0.1625,0.7911 ± 0.0952
train_recall,0.7961 ± 0.0743,0.8229 ± 0.0193,0.8227 ± 0.0389,0.8123 ± 0.0354
test_f1,0.6733 ± 0.0697,0.6891 ± 0.0324,0.6741 ± 0.0550,0.6848 ± 0.0777
train_f1,0.7413 ± 0.0536,0.7474 ± 0.0263,0.7618 ± 0.0256,0.7722 ± 0.0298


Results for Decision Tree:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5389 ± 0.1647,0.4935 ± 0.0843,0.5612 ± 0.1375,0.5584 ± 0.1561
train_roc_auc,0.7941 ± 0.0230,0.7967 ± 0.0375,0.7533 ± 0.0529,0.8135 ± 0.0169
test_precision,0.6053 ± 0.0779,0.5350 ± 0.0369,0.6525 ± 0.0808,0.6240 ± 0.1196
train_precision,0.8088 ± 0.0444,0.7810 ± 0.0908,0.7926 ± 0.1063,0.8591 ± 0.0553
test_recall,0.4800 ± 0.1041,0.7556 ± 0.1847,0.8133 ± 0.1529,0.6444 ± 0.0855
train_recall,0.7808 ± 0.1176,0.8480 ± 0.1390,0.8271 ± 0.1620,0.7493 ± 0.1190
test_f1,0.5260 ± 0.0755,0.6185 ± 0.0801,0.7065 ± 0.0207,0.6283 ± 0.0930
train_f1,0.7869 ± 0.0539,0.7975 ± 0.0529,0.7877 ± 0.0485,0.7907 ± 0.0458


Results for Random Forest:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6954 ± 0.0859,0.5265 ± 0.1134,0.5253 ± 0.0668,0.5231 ± 0.1681
train_roc_auc,0.9219 ± 0.0195,0.9392 ± 0.0196,0.9634 ± 0.0128,0.9560 ± 0.0065
test_precision,0.6603 ± 0.0847,0.5614 ± 0.0300,0.5729 ± 0.0380,0.5798 ± 0.0573
train_precision,0.7910 ± 0.0100,0.8160 ± 0.0747,0.8396 ± 0.0452,0.8248 ± 0.0363
test_recall,0.8067 ± 0.1143,0.8778 ± 0.0744,0.8156 ± 0.1137,0.7733 ± 0.0947
train_recall,0.9269 ± 0.0308,0.9478 ± 0.0290,0.9843 ± 0.0128,0.9633 ± 0.0357
test_f1,0.7243 ± 0.0920,0.6829 ± 0.0273,0.6701 ± 0.0580,0.6597 ± 0.0592
train_f1,0.8534 ± 0.0167,0.8749 ± 0.0452,0.9053 ± 0.0236,0.8874 ± 0.0160


## ICANS >=2

In [49]:
y = targets["ae_summ_icans_highestgrade_v2"]

In [50]:
y = y.astype('int8')

In [51]:
y[y<2] = 0
y[y>=2] = 1


In [52]:
y.value_counts()

ae_summ_icans_highestgrade_v2
0    52
1    33
Name: count, dtype: int64

In [53]:
icans_2 = result_viewer(preprocessor, datasets, y)

Results for Logistic Regression:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5861 ± 0.0763,0.4993 ± 0.1030,0.4928 ± 0.0899,0.6128 ± 0.1947
train_roc_auc,0.7941 ± 0.0231,0.7811 ± 0.0288,0.7649 ± 0.0523,0.7736 ± 0.0538
test_precision,0.5000 ± 0.1054,0.4333 ± 0.2305,0.5667 ± 0.2494,0.6267 ± 0.1619
train_precision,0.6749 ± 0.0427,0.7795 ± 0.0785,0.7524 ± 0.0858,0.7304 ± 0.0551
test_recall,0.3381 ± 0.1931,0.3333 ± 0.2172,0.2143 ± 0.0797,0.5238 ± 0.2151
train_recall,0.5299 ± 0.0487,0.5678 ± 0.0484,0.4920 ± 0.1265,0.4920 ± 0.1127
test_f1,0.3873 ± 0.1665,0.3615 ± 0.2006,0.2893 ± 0.0836,0.5525 ± 0.2029
train_f1,0.5921 ± 0.0376,0.6548 ± 0.0461,0.5872 ± 0.1109,0.5813 ± 0.0890


Results for Decision Tree:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6349 ± 0.0271,0.5792 ± 0.0316,0.6306 ± 0.0303,0.4927 ± 0.0989
train_roc_auc,0.7532 ± 0.0355,0.7729 ± 0.0560,0.7810 ± 0.0436,0.7682 ± 0.0362
test_precision,0.5923 ± 0.2093,0.5330 ± 0.2353,0.7111 ± 0.2367,0.3200 ± 0.2638
train_precision,0.6686 ± 0.1179,0.7066 ± 0.1144,0.8471 ± 0.0991,0.8008 ± 0.1440
test_recall,0.5476 ± 0.3493,0.5571 ± 0.3270,0.3571 ± 0.1869,0.1810 ± 0.1741
train_recall,0.7433 ± 0.2840,0.7168 ± 0.2613,0.5083 ± 0.1226,0.5481 ± 0.2393
test_f1,0.4559 ± 0.1664,0.4424 ± 0.1406,0.4238 ± 0.1133,0.2244 ± 0.2037
train_f1,0.6466 ± 0.0956,0.6611 ± 0.1057,0.6183 ± 0.0633,0.5973 ± 0.1309


Results for Random Forest:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6623 ± 0.1067,0.5412 ± 0.0986,0.6290 ± 0.0306,0.5972 ± 0.0367
train_roc_auc,0.9430 ± 0.0103,0.9563 ± 0.0098,0.9675 ± 0.0133,0.9725 ± 0.0111
test_precision,0.5000 ± 0.4472,0.2833 ± 0.2769,0.3333 ± 0.3651,0.4000 ± 0.3742
train_precision,0.9750 ± 0.0500,1.0000 ± 0.0000,0.9857 ± 0.0286,0.9769 ± 0.0462
test_recall,0.0857 ± 0.0700,0.1571 ± 0.1829,0.1524 ± 0.1393,0.0905 ± 0.0744
train_recall,0.4402 ± 0.1006,0.4621 ± 0.0739,0.5689 ± 0.1009,0.5838 ± 0.1579
test_f1,0.1444 ± 0.1184,0.2000 ± 0.2191,0.2015 ± 0.1906,0.1460 ± 0.1215
train_f1,0.5976 ± 0.0945,0.6286 ± 0.0698,0.7171 ± 0.0779,0.7150 ± 0.0944


# CRS 0,1

In [54]:
y = targets["ae_summ_crs_v2"]

In [55]:
y.value_counts()

ae_summ_crs_v2
1.0    79
0.0     6
Name: count, dtype: int64

In [56]:
crs_1 = result_viewer(preprocessor, datasets, y)

Results for Logistic Regression:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5917 ± 0.3208,0.6133 ± 0.0914,0.5200 ± 0.2372,0.6292 ± 0.2406
train_roc_auc,0.9222 ± 0.0424,0.9205 ± 0.0474,0.9180 ± 0.0280,0.9165 ± 0.0080
test_precision,0.9294 ± 0.0235,0.9294 ± 0.0235,0.9412 ± 0.0372,0.9279 ± 0.0229
train_precision,0.9294 ± 0.0059,0.9294 ± 0.0059,0.9294 ± 0.0059,0.9322 ± 0.0070
test_recall,1.0000 ± 0.0000,1.0000 ± 0.0000,0.9000 ± 0.2000,0.9750 ± 0.0306
train_recall,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000
test_f1,0.9633 ± 0.0129,0.9633 ± 0.0129,0.9027 ± 0.1186,0.9504 ± 0.0158
train_f1,0.9634 ± 0.0031,0.9634 ± 0.0031,0.9634 ± 0.0031,0.9649 ± 0.0037


Results for Decision Tree:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.4179 ± 0.2231,0.7308 ± 0.2505,0.5188 ± 0.1839,0.5117 ± 0.1554
train_roc_auc,0.8927 ± 0.0250,0.9061 ± 0.0246,0.9167 ± 0.0598,0.8686 ± 0.0875
test_precision,0.9254 ± 0.0219,0.9404 ± 0.0015,0.9404 ± 0.0015,0.9253 ± 0.0220
train_precision,0.9488 ± 0.0165,0.9433 ± 0.0140,0.9607 ± 0.0235,0.9461 ± 0.0142
test_recall,0.9375 ± 0.0685,1.0000 ± 0.0000,1.0000 ± 0.0000,0.9375 ± 0.0791
train_recall,0.9841 ± 0.0201,0.9969 ± 0.0063,0.9874 ± 0.0119,0.9937 ± 0.0127
test_f1,0.9298 ± 0.0339,0.9693 ± 0.0008,0.9693 ± 0.0008,0.9294 ± 0.0398
train_f1,0.9658 ± 0.0041,0.9693 ± 0.0047,0.9736 ± 0.0091,0.9692 ± 0.0047


Results for Random Forest:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.7821 ± 0.1221,0.6125 ± 0.2512,0.6667 ± 0.3124,0.6200 ± 0.2407
train_roc_auc,0.9642 ± 0.0242,0.9804 ± 0.0025,0.9867 ± 0.0097,0.9911 ± 0.0109
test_precision,0.9294 ± 0.0235,0.9294 ± 0.0235,0.9294 ± 0.0235,0.9294 ± 0.0235
train_precision,0.9322 ± 0.0070,0.9294 ± 0.0059,0.9350 ± 0.0113,0.9294 ± 0.0059
test_recall,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000
train_recall,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000
test_f1,0.9633 ± 0.0129,0.9633 ± 0.0129,0.9633 ± 0.0129,0.9633 ± 0.0129
train_f1,0.9649 ± 0.0037,0.9634 ± 0.0031,0.9664 ± 0.0060,0.9634 ± 0.0031


# CRS >=2

In [57]:
y = targets["ae_summ_highestgrade_v2"]
y = y.astype('int8')
y[y<2] = 0
y[y>=2] = 1

y.value_counts()

ae_summ_highestgrade_v2
0    55
1    30
Name: count, dtype: int64

In [58]:
crs_2 = result_viewer(preprocessor, datasets, y)

Results for Logistic Regression:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6030 ± 0.1567,0.7455 ± 0.1340,0.6606 ± 0.1628,0.5606 ± 0.1324
train_roc_auc,0.7595 ± 0.0532,0.8322 ± 0.0364,0.7998 ± 0.0590,0.8400 ± 0.0358
test_precision,0.3200 ± 0.2841,0.5200 ± 0.3357,0.5617 ± 0.2634,0.3167 ± 0.0972
train_precision,0.5974 ± 0.0875,0.7245 ± 0.0833,0.6831 ± 0.0782,0.7329 ± 0.0830
test_recall,0.2000 ± 0.1944,0.3333 ± 0.1826,0.4000 ± 0.2494,0.2667 ± 0.1333
train_recall,0.3417 ± 0.0808,0.5167 ± 0.1137,0.4833 ± 0.1594,0.5750 ± 0.1546
test_f1,0.2424 ± 0.2239,0.3980 ± 0.2274,0.4364 ± 0.1974,0.2816 ± 0.1136
train_f1,0.4330 ± 0.0864,0.6001 ± 0.1029,0.5584 ± 0.1395,0.6379 ± 0.1292


Results for Decision Tree:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5000 ± 0.1002,0.5379 ± 0.1325,0.5182 ± 0.0912,0.6045 ± 0.1512
train_roc_auc,0.7205 ± 0.0361,0.8096 ± 0.0304,0.7673 ± 0.0336,0.8144 ± 0.0158
test_precision,0.0667 ± 0.1333,0.2333 ± 0.2906,0.5000 ± 0.2789,0.3969 ± 0.2099
train_precision,0.8914 ± 0.0911,0.9024 ± 0.0913,0.9458 ± 0.0741,0.7568 ± 0.1623
test_recall,0.0333 ± 0.0667,0.1000 ± 0.1333,0.2000 ± 0.0667,0.4667 ± 0.2867
train_recall,0.2333 ± 0.0773,0.4583 ± 0.1807,0.3917 ± 0.1007,0.6083 ± 0.2351
test_f1,0.0444 ± 0.0889,0.1389 ± 0.1809,0.2778 ± 0.1152,0.4086 ± 0.2127
train_f1,0.3615 ± 0.0974,0.5757 ± 0.1336,0.5413 ± 0.0876,0.6173 ± 0.0918


Results for Random Forest:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5273 ± 0.1357,0.5909 ± 0.1362,0.6121 ± 0.0625,0.5424 ± 0.2103
train_roc_auc,0.8842 ± 0.0245,0.9326 ± 0.0050,0.9150 ± 0.0192,0.9718 ± 0.0198
test_precision,0.4000 ± 0.4899,0.5333 ± 0.3232,0.3476 ± 0.2237,0.1000 ± 0.2000
train_precision,0.9818 ± 0.0364,0.9486 ± 0.0742,0.8874 ± 0.0382,0.9713 ± 0.0353
test_recall,0.1000 ± 0.1333,0.2000 ± 0.1247,0.2667 ± 0.2261,0.0333 ± 0.0667
train_recall,0.2583 ± 0.1067,0.5417 ± 0.1021,0.5333 ± 0.0612,0.4833 ± 0.1137
test_f1,0.1571 ± 0.2040,0.2760 ± 0.1554,0.2864 ± 0.2092,0.0500 ± 0.1000
train_f1,0.3959 ± 0.1296,0.6786 ± 0.0612,0.6649 ± 0.0572,0.6363 ± 0.1013


# Outcome
CMR vs others

In [59]:
y = targets["surv_bestresponse_car"]

In [60]:
y.value_counts()

surv_bestresponse_car
1.0    63
2.0    12
4.0     8
0.0     2
Name: count, dtype: int64

Early Death: 0, CMR: 1, PMR:2, SD: 3, PD:4

In [61]:
y = y.astype('int8')

In [62]:
y[y != 1] = 0

In [63]:
y.value_counts()

surv_bestresponse_car
1    63
0    22
Name: count, dtype: int64

In [64]:
outcome = result_viewer(preprocessor, datasets, y)

Results for Logistic Regression:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6628 ± 0.1497,0.6244 ± 0.1303,0.6336 ± 0.1411,0.6244 ± 0.1206
train_roc_auc,0.8293 ± 0.0330,0.8053 ± 0.0464,0.8203 ± 0.0280,0.8494 ± 0.0470
test_precision,0.7897 ± 0.0564,0.7810 ± 0.0463,0.7337 ± 0.0372,0.7909 ± 0.0716
train_precision,0.8053 ± 0.0343,0.8124 ± 0.0346,0.8263 ± 0.0228,0.8079 ± 0.0256
test_recall,0.8885 ± 0.0630,0.9064 ± 0.0895,0.8744 ± 0.0597,0.8577 ± 0.0894
train_recall,0.9485 ± 0.0340,0.9484 ± 0.0204,0.9603 ± 0.0127,0.9285 ± 0.0201
test_f1,0.8360 ± 0.0581,0.8369 ± 0.0536,0.7968 ± 0.0384,0.8168 ± 0.0326
train_f1,0.8708 ± 0.0308,0.8742 ± 0.0116,0.8881 ± 0.0156,0.8636 ± 0.0142


Results for Decision Tree:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5860 ± 0.0598,0.5297 ± 0.0354,0.4801 ± 0.0797,0.5203 ± 0.0711
train_roc_auc,0.7572 ± 0.0614,0.8189 ± 0.0411,0.8163 ± 0.0437,0.8359 ± 0.0274
test_precision,0.7752 ± 0.0284,0.7667 ± 0.0279,0.7459 ± 0.0213,0.7658 ± 0.0324
train_precision,0.8458 ± 0.0381,0.8744 ± 0.0514,0.8761 ± 0.0433,0.8390 ± 0.0365
test_recall,0.8859 ± 0.1238,0.8872 ± 0.0686,0.8397 ± 0.0926,0.8744 ± 0.1433
train_recall,0.9365 ± 0.0463,0.9288 ± 0.0364,0.9446 ± 0.0627,0.9803 ± 0.0175
test_f1,0.8240 ± 0.0695,0.8213 ± 0.0393,0.7877 ± 0.0480,0.8092 ± 0.0669
train_f1,0.8871 ± 0.0156,0.8987 ± 0.0120,0.9061 ± 0.0138,0.9035 ± 0.0170


Results for Random Forest:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6962 ± 0.1075,0.6144 ± 0.2112,0.5762 ± 0.1282,0.6059 ± 0.0591
train_roc_auc,0.9420 ± 0.0255,0.9508 ± 0.0108,0.9380 ± 0.0185,0.9692 ± 0.0110
test_precision,0.7541 ± 0.0303,0.7534 ± 0.0368,0.7501 ± 0.0589,0.7480 ± 0.0454
train_precision,0.7953 ± 0.0160,0.8218 ± 0.0310,0.8426 ± 0.0203,0.8264 ± 0.0141
test_recall,0.9692 ± 0.0377,0.9679 ± 0.0393,0.9372 ± 0.0315,0.9372 ± 0.0900
train_recall,1.0000 ± 0.0000,1.0000 ± 0.0000,0.9961 ± 0.0078,1.0000 ± 0.0000
test_f1,0.8472 ± 0.0164,0.8467 ± 0.0308,0.8315 ± 0.0330,0.8291 ± 0.0478
train_f1,0.8859 ± 0.0099,0.9019 ± 0.0188,0.9128 ± 0.0130,0.9049 ± 0.0084
